In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
import warnings
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
warnings.filterwarnings('ignore')


# 1. 데이터 로딩
cust_df = pd.read_csv("../data/santander-customer-satisfaction/train.csv", encoding='latin-1')
print('dataset shape:', cust_df.shape)
cust_df.head(3)
 

dataset shape: (76020, 371)


,ID,var3,var15,imp_ent_var16_ult1,imp_op_var39_comer_ult1,imp_op_var39_comer_ult3,imp_op_var40_comer_ult1,imp_op_var40_comer_ult3,imp_op_var40_efect_ult1,imp_op_var40_efect_ult3,...,saldo_medio_var33_hace2,saldo_medio_var33_hace3,saldo_medio_var33_ult1,saldo_medio_var33_ult3,saldo_medio_var44_hace2,saldo_medio_var44_hace3,saldo_medio_var44_ult1,saldo_medio_var44_ult3,var38,TARGET
0,1,2,23,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,39205.17,0
1,3,2,34,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,49278.03,0
2,4,2,23,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,67333.77,0


In [3]:
cust_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 76020 entries, 0 to 76019
Columns: 371 entries, ID to TARGET
dtypes: float64(111), int64(260)
memory usage: 215.2 MB


In [4]:
# 2. 불균형 확인
print(cust_df['TARGET'].value_counts())
unsatisfied_cnt = cust_df[cust_df['TARGET'] == 1].TARGET.count()
total_cnt = cust_df.TARGET.count()
print('unsatisfied 비율은 {0:.2f}'.format((unsatisfied_cnt / total_cnt)))

TARGET
0    73012
1     3008
Name: count, dtype: int64
unsatisfied 비율은 0.04


In [5]:
cust_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 76020 entries, 0 to 76019
Columns: 371 entries, ID to TARGET
dtypes: float64(111), int64(260)
memory usage: 215.2 MB


In [6]:
# 3. 이상값 탐지(var3의 min: -999999)
cust_df.describe()

,ID,var3,var15,imp_ent_var16_ult1,imp_op_var39_comer_ult1,imp_op_var39_comer_ult3,imp_op_var40_comer_ult1,imp_op_var40_comer_ult3,imp_op_var40_efect_ult1,imp_op_var40_efect_ult3,...,saldo_medio_var33_hace2,saldo_medio_var33_hace3,saldo_medio_var33_ult1,saldo_medio_var33_ult3,saldo_medio_var44_hace2,saldo_medio_var44_hace3,saldo_medio_var44_ult1,saldo_medio_var44_ult3,var38,TARGET
count,76020.000000,76020.000000,76020.000000,76020.000000,76020.000000,76020.000000,76020.000000,76020.000000,76020.000000,76020.000000,...,76020.000000,76020.000000,76020.000000,76020.000000,76020.000000,76020.000000,76020.000000,76020.000000,7.602000e+04,76020.000000
mean,75964.050723,-1523.199277,33.212865,86.208265,72.363067,119.529632,3.559130,6.472698,0.412946,0.567352,...,7.935824,1.365146,12.215580,8.784074,31.505324,1.858575,76.026165,56.614351,1.172358e+05,0.039569
std,43781.947379,39033.462364,12.956486,1614.757313,339.315831,546.266294,93.155749,153.737066,30.604864,36.513513,...,455.887218,113.959637,783.207399,538.439211,2013.125393,147.786584,4040.337842,2852.579397,1.826646e+05,0.194945
min,1.000000,-999999.000000,5.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,5.163750e+03,0.000000
25%,38104.750000,2.000000,23.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,6.787061e+04,0.000000
50%,76043.000000,2.000000,28.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.064092e+05,0.000000
75%,113748.750000,2.000000,40.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.187563e+05,0.000000
max,151838.000000,238.000000,105.000000,210000.000000,12888.030000,21024.810000,8237.820000,11073.570000,6600.000000,6600.000000,...,50003.880000,20385.720000,138831.630000,91778.730000,438329.220000,24650.010000,681462.900000,397884.300000,2.203474e+07,1.000000


In [7]:
# 4-1. 전처리(var3의 -999999 -> 2  & ID 제거)
cust_df["var3"] = cust_df["var3"].replace(-999999,2)
cust_df.drop("ID", axis=1, inplace=True)

In [8]:
# 일반 데이터와 레이블 분리
X_features = cust_df.iloc[:,:-1]
y_labels = cust_df.iloc[:,-1]

In [9]:
# 4-2. 전처리(분산 0인 컬럼 제거)
# 1. 각 컬럼의 분산 계산
stds = X_features.std()

# 2. 분산이 0인(즉, 표준편차가 0이거나 값이 모두 똑같은) 컬럼 이름 추출
zero_var_cols = stds[stds == 0].index.tolist()

print(f"분산이 0인 컬럼 개수: {len(zero_var_cols)}")
print(f"삭제할 컬럼들: {zero_var_cols}")

# 3. 해당 컬럼들 제거
X_features_clean = X_features.drop(columns=zero_var_cols)

print(f"정제 전 피처 shape: {X_features.shape}")
print(f"정제 후 피처 shape: {X_features_clean.shape}")

분산이 0인 컬럼 개수: 34
삭제할 컬럼들: ['ind_var2_0', 'ind_var2', 'ind_var27_0', 'ind_var28_0', 'ind_var28', 'ind_var27', 'ind_var41', 'ind_var46_0', 'ind_var46', 'num_var27_0', 'num_var28_0', 'num_var28', 'num_var27', 'num_var41', 'num_var46_0', 'num_var46', 'saldo_var28', 'saldo_var27', 'saldo_var41', 'saldo_var46', 'imp_amort_var18_hace3', 'imp_amort_var34_hace3', 'imp_reemb_var13_hace3', 'imp_reemb_var33_hace3', 'imp_trasp_var17_out_hace3', 'imp_trasp_var33_out_hace3', 'num_var2_0_ult1', 'num_var2_ult1', 'num_reemb_var13_hace3', 'num_reemb_var33_hace3', 'num_trasp_var17_out_hace3', 'num_trasp_var33_out_hace3', 'saldo_var2_ult1', 'saldo_medio_var13_medio_hace3']
정제 전 피처 shape: (76020, 369)
정제 후 피처 shape: (76020, 335)


In [10]:
# 값이 완전히 동일한 컬럼 중 뒤에 나온 것들의 이름을 반환
import numpy as np

def find_duplicate_columns(df):

    groups = {}
    for col in df.columns:
        v = df[col].values
        # 지문(sum/min/max)으로 후보를 먼저 좁힘
        key = (v.sum(), v.min(), v.max())
        groups.setdefault(key, []).append(col)

    dup = set()
    for cols in groups.values():
        if len(cols) < 2:
            continue
        for i in range(len(cols)):
            if cols[i] in dup:
                continue
            for j in range(i + 1, len(cols)):
                if cols[j] in dup:
                    continue
                if np.array_equal(df[cols[i]].values, df[cols[j]].values):
                    dup.add(cols[j])
    return sorted(dup)


dup_cols = find_duplicate_columns(X_features_clean)
print(f"중복 컬럼 개수: {len(dup_cols)}")
print(f"삭제할 컬럼들: {dup_cols}")

X_features_clean = X_features_clean.drop(columns=dup_cols)
print(f"정제 후 피처 shape: {X_features_clean.shape}")

중복 컬럼 개수: 29
삭제할 컬럼들: ['delta_num_reemb_var13_1y3', 'delta_num_reemb_var17_1y3', 'delta_num_reemb_var33_1y3', 'delta_num_trasp_var17_in_1y3', 'delta_num_trasp_var17_out_1y3', 'delta_num_trasp_var33_in_1y3', 'delta_num_trasp_var33_out_1y3', 'ind_var13_medio', 'ind_var18', 'ind_var25', 'ind_var26', 'ind_var29', 'ind_var29_0', 'ind_var32', 'ind_var34', 'ind_var37', 'ind_var39', 'num_var13_medio', 'num_var18', 'num_var25', 'num_var26', 'num_var29', 'num_var29_0', 'num_var32', 'num_var34', 'num_var37', 'num_var39', 'saldo_medio_var13_medio_ult1', 'saldo_var29']
정제 후 피처 shape: (76020, 306)


In [11]:
# =========================================================================
# 4. 데이터셋 단일 분할 및 대조 실험군 독립 분기 (Hold-out 공정성 확보)
# =========================================================================
from sklearn.metrics import recall_score
from xgboost import XGBClassifier


print("\n=== 데이터 분할 및 대조 실험군 독립 분기 ===")

X_train, X_test, y_train, y_test = train_test_split(
    X_features_clean, y_labels, test_size=0.2, random_state=156, stratify=y_labels
)

# [실험 A용 데이터셋 준비] 원본 버전 (var38 수치가 로그 없이 날것으로 유지됨)
X_train_norm = X_train.copy()
X_test_norm = X_test.copy()

# [실험 B용 데이터셋 준비] 로그 변환 버전 🚀
X_train_log = X_train.copy()
X_test_log = X_test.copy()

# 실험 B 세트 내부의 'var38'에만 정밀하게 로그 트랜스포메이션 주입
X_train_log['var38'] = np.log1p(X_train_log['var38'].astype(float))
X_test_log['var38'] = np.log1p(X_test_log['var38'].astype(float))

print("  - 데이터 복사 타입 충돌 및 카테고리 불일치 에러 완전 방어 완료!")


# =========================================================================
# 5. 최신 XGBoost 인프라 정의 및 대조군 학습/평가
# =========================================================================
print("\n=== XGBoost 비교 학습 및 최종 스코어 측정 ===")

def get_xgb_model():
    return XGBClassifier(
        n_estimators=500,
        learning_rate=0.05,
        max_depth=5,
        min_child_weight=5,
        subsample=0.8,
        colsample_bytree=0.8,
        early_stopping_rounds=100,
        eval_metric='auc',
        # 원-핫 인코딩을 완료하여 모두 수치형 칼럼으로 변환되었으므로 enable_categorical 옵션은 제외합니다.
        random_state=156
    )

# -------------------------------------------------------------------------
# 실험 A: 원본 var38 데이터셋 기반 학습 진행
# -------------------------------------------------------------------------
print("  - 1. [실험 A] var38 원본 데이터셋 기반 모델 학습 중...")
xgb_norm = get_xgb_model()
xgb_norm.fit(
    X_train_norm, y_train,
    eval_set=[(X_train_norm, y_train), (X_test_norm, y_test)],
    verbose=False
)
norm_pred_proba = xgb_norm.predict_proba(X_test_norm)[:, 1]
auc_normal = roc_auc_score(y_test, norm_pred_proba)

# 🚀 정확도 및 재현율 계산용 예측값 추출
norm_preds = xgb_norm.predict(X_test_norm)
acc_normal = accuracy_score(y_test, norm_preds)
rec_normal = recall_score(y_test, norm_preds)

# -------------------------------------------------------------------------
# 실험 B: 로그 변환 var38 데이터셋 기반 학습 진행
# -------------------------------------------------------------------------
print("  - 2. [실험 B] var38 로그 변환 데이터셋 기반 모델 학습 중...")
xgb_log = get_xgb_model()
xgb_log.fit(
    X_train_log, y_train,
    eval_set=[(X_train_log, y_train), (X_test_log, y_test)],
    verbose=False
)
log_pred_proba = xgb_log.predict_proba(X_test_log)[:, 1]
auc_log = roc_auc_score(y_test, log_pred_proba)

# 🚀 정확도 및 재현율 계산용 예측값 추출
log_preds = xgb_log.predict(X_test_log)
acc_log = accuracy_score(y_test, log_preds)
rec_log = recall_score(y_test, log_preds)


# =========================================================================
# 6. 최종 분석 결과 종합 출력
# =========================================================================
print("\n" + "="*65)
print("             [ 최종 분석 논문 검증 결과 종합 성적표 ]")
print("="*65)
print("  구분                 |   ROC-AUC   |   정확도    |   재현율   ")
print("-"*65)
print(f"  [실험 A] 로그 전    |    {auc_normal:.4f}    |    {acc_normal:.4f}    |   {rec_normal:.4f}")
print(f"  [실험 B] 로그 후    |    {auc_log:.4f}    |    {acc_log:.4f}    |   {rec_log:.4f}")
print("-"*65)
print(f"  순수 개선 편차       |   {auc_log - auc_normal:+.4f}    |   {acc_log - acc_normal:+.4f}    |   {rec_log - rec_normal:+.4f}")
print("="*65)

# 영찬 님 결과
# 결과 : ROC AUC : 0.8517(컬럼정제전)
# 결과 : ROC AUC : 0.8531(컬럼정제후)

# 내 결과
# =================================================================
#              [ 최종 분석 논문 검증 결과 종합 성적표 ]
# =================================================================
#   구분                 |   ROC-AUC   |   정확도    |   재현율   
# -----------------------------------------------------------------
#   [실험 A] 로그 전    |    0.8520    |    0.9606    |   0.0066
#   [실험 B] 로그 후    |    0.8520    |    0.9606    |   0.0066
# -----------------------------------------------------------------
#   순수 개선 편차       |   -0.0000    |   +0.0000    |   +0.0000
# =================================================================


=== 데이터 분할 및 대조 실험군 독립 분기 ===
  - 데이터 복사 타입 충돌 및 카테고리 불일치 에러 완전 방어 완료!

=== XGBoost 비교 학습 및 최종 스코어 측정 ===
  - 1. [실험 A] var38 원본 데이터셋 기반 모델 학습 중...
  - 2. [실험 B] var38 로그 변환 데이터셋 기반 모델 학습 중...

             [ 최종 분석 논문 검증 결과 종합 성적표 ]
  구분                 |   ROC-AUC   |   정확도    |   재현율   
-----------------------------------------------------------------
  [실험 A] 로그 전    |    0.8520    |    0.9606    |   0.0066
  [실험 B] 로그 후    |    0.8520    |    0.9606    |   0.0066
-----------------------------------------------------------------
  순수 개선 편차       |   -0.0000    |   +0.0000    |   +0.0000


---

## [담당자: 정찬성] XGBoost 외 4개 모델(RandomForest/LogisticRegression/LightGBM/GradientBoost) 비교 실습

위 §5(XGBoost 비교 학습 및 최종 스코어 측정, cell 9)까지가 기존에 완료된 내용이다 — Santander 데이터(76,020행, 정제 후 306개 피처)를 `X_train`/`X_test`(80/20, stratify)로 한 번만 분할한 뒤, 그 위에서 **실험 A(`var38` 원본)**와 **실험 B(`var38`을 `log1p` 변환)** 두 대조군을 XGBoost로 각각 학습·평가했다. 이 지점부터는 **완전히 동일한 `X_train_norm`/`X_test_norm`/`X_train_log`/`X_test_log`/`y_train`/`y_test`(위 cell 9에서 이미 만든 것을 재사용, 새로 분할하지 않음)**로 나머지 4개 모델(RandomForest, LogisticRegression, LightGBM, GradientBoost)을 XGBoost와 동일한 실험 A/B 구조로 돌려본다.

```mermaid
flowchart TD
    LOAD["1~4. 데이터 로드/정제(위, 완료)<br/>var3 이상치 처리 · ID 제거 · 분산0/중복 컬럼 제거<br/>(76,020행 x 306피처)"] --> SPLIT["Hold-out 단일 분할(위, 완료)<br/>X_train/X_test 80/20, stratify=y_labels"]
    SPLIT --> EXPA["실험 A: var38 원본(log 미적용)<br/>X_train_norm / X_test_norm"]
    SPLIT --> EXPB["실험 B: var38 → log1p 변환<br/>X_train_log / X_test_log"]

    EXPA --> XGB["① XGBoost(완료, 위 cell 9)"]
    EXPB --> XGB
    EXPA --> RF["② RandomForest(신규)"]
    EXPB --> RF
    EXPA --> LR["③ LogisticRegression(신규)"]
    EXPB --> LR
    EXPA --> LGBM["④ LightGBM(신규)"]
    EXPB --> LGBM
    EXPA --> GB["⑤ GradientBoost(신규)"]
    EXPB --> GB

    XGB --> CMP["최종 종합 비교<br/>5개 모델 x 실험 A/B = 10행"]
    RF --> CMP
    LR --> CMP
    LGBM --> CMP
    GB --> CMP

    style XGB fill:#E4F5E9,stroke:#2E8B57
    style CMP fill:#E0ECFF,stroke:#2E6DE5
```

| 모델 | 하이퍼파라미터 | XGBoost(n_estimators=500, learning_rate=0.05, max_depth=5, min_child_weight=5, subsample=0.8, colsample_bytree=0.8, early_stopping_rounds=100)와 대응 관계 |
|---|---|---|
| RandomForest | `n_estimators=500, max_depth=5, min_samples_leaf=5, max_features=0.8, max_samples=0.8, n_jobs=-1, random_state=156` | `max_features`가 `colsample_bytree`(분할마다 피처 샘플링 비율)에, `max_samples`가 `subsample`(트리마다 행 샘플링 비율)에, `min_samples_leaf`가 `min_child_weight`(리프의 최소 표본 조건)에 대응한다. 단, RandomForest는 배깅(bagging) 계열이라 "이전 트리의 오차를 보정"하는 개념이 없어 **조기종료(early stopping) 자체가 존재하지 않는다** — 트리 500개를 전부 채운다. |
| LogisticRegression | `LogisticRegression(max_iter=1000, random_state=156)` | 선형모델이라 트리·부스팅 관련 파라미터가 전혀 대응되지 않는다. 피처가 306개로 비교적 많아 기본 반복횟수(100)로는 수렴 경고가 날 수 있어 `max_iter=1000`으로 여유를 뒀다. |
| LightGBM | `n_estimators=500, learning_rate=0.05, max_depth=5, min_child_samples=5, subsample=0.8, colsample_bytree=0.8, random_state=156` + `eval_X`/`eval_y`/`callbacks=[lgb.early_stopping(100)]` | XGBoost와 파라미터 이름·값을 거의 그대로 맞췄다(`min_child_weight` → LightGBM에서는 `min_child_samples`가 같은 역할). 조기종료도 `early_stopping_rounds=100`과 동일한 의미로 `stopping_rounds=100`을 콜백으로 지정했다 — LightGBM 최신 sklearn API는 `eval_set` 대신 `eval_X`/`eval_y`를 쓴다. |
| GradientBoost | `n_estimators=500, learning_rate=0.05, max_depth=5, subsample=0.8, random_state=156, n_iter_no_change=100, validation_fraction=0.1` | XGBoost의 원조 알고리즘이라 가장 직접 비교되는 짝이다. `colsample_bytree`/`min_child_weight`에 대응하는 옵션이 sklearn `GradientBoostingClassifier`에는 없다(한계로 명시). 조기종료는 `early_stopping_rounds`처럼 별도 검증셋을 넘기는 방식이 아니라, 학습 데이터의 `validation_fraction`(10%)을 내부적으로 떼어 `n_iter_no_change=100`(100라운드 개선 없으면 중단) 기준으로 판단한다. |

In [12]:
# 기술적 의미: XGBoost 실험(§cell 9)은 "모델 학습 → 예측 → AUC/정확도/재현율 계산 → 성적표 출력"을 실험 A/B마다 손으로 반복 작성했다.
# 왜: 이 패턴을 4개 모델 x 2개 실험 = 8번 그대로 복사하면 코드가 지나치게 길어지고 오탈자 위험이 커진다. §cell 9의 로직(모델 학습 → predict_proba/predict → roc_auc_score/accuracy_score/recall_score)을 그대로 함수로 옮겨 재사용하되, §cell 9 자체는 건드리지 않는다(기존 XGBoost 실측 결과 보존).
from sklearn.metrics import accuracy_score, roc_auc_score, recall_score

# 기술적 의미: 이미 생성된 모델 객체와 학습/검증 데이터, 그리고 모델별로 다른 fit() 추가 인자(**fit_kwargs, 예: LightGBM의 조기종료 콜백)를 받아 학습·예측·평가까지 한 번에 수행하는 함수를 정의한다.
# 업무적 의미: RandomForest/LogisticRegression처럼 추가 인자가 필요 없는 모델과 LightGBM처럼 조기종료 콜백이 필요한 모델을 같은 함수 하나로 통일해서 다룰 수 있게 한다.
def run_experiment(model, X_tr, y_tr, X_te, y_te, **fit_kwargs):
    # 기술적 의미: 학습 데이터와 모델별 추가 인자(fit_kwargs)로 모델을 학습시킨다.
    model.fit(X_tr, y_tr, **fit_kwargs)
    # 기술적 의미: 검증 데이터에 대한 예측 확률(클래스 1=불만족 고객일 확률)을 구한다 — AUC 계산에 필요하다.
    pred_proba = model.predict_proba(X_te)[:, 1]
    # 기술적 의미: 검증 데이터에 대한 최종 예측 클래스(0/1)를 구한다 — 정확도·재현율 계산에 필요하다.
    pred = model.predict(X_te)
    # 기술적 의미: ROC-AUC, 정확도, 재현율 3개 지표를 dict로 만들어 반환한다. §cell 9와 동일한 3개 지표다.
    metrics = {
        'auc': roc_auc_score(y_te, pred_proba),
        'acc': accuracy_score(y_te, pred),
        'rec': recall_score(y_te, pred),
    }
    # 기술적 의미: 학습이 끝난 모델 객체와 지표 dict를 함께 반환한다 — 모델 객체는 이후 feature_importances_/coef_ 조회에 쓰인다.
    return model, metrics


# 기술적 의미: §cell 9의 "최종 분석 논문 검증 결과 종합 성적표" 출력 블록과 완전히 동일한 서식으로, 모델명만 바꿔 출력하는 함수를 정의한다.
# 업무적 의미: 4개 모델 모두 XGBoost와 시각적으로 1:1 비교 가능한 동일한 표 형식으로 결과를 남긴다.
def print_scorecard(model_name, metrics_norm, metrics_log):
    print("\n" + "="*65)
    print(f"        [ {model_name} 최종 분석 논문 검증 결과 종합 성적표 ]")
    print("="*65)
    print("  구분                 |   ROC-AUC   |   정확도    |   재현율   ")
    print("-"*65)
    print(f"  [실험 A] 로그 전    |    {metrics_norm['auc']:.4f}    |    {metrics_norm['acc']:.4f}    |   {metrics_norm['rec']:.4f}")
    print(f"  [실험 B] 로그 후    |    {metrics_log['auc']:.4f}    |    {metrics_log['acc']:.4f}    |   {metrics_log['rec']:.4f}")
    print("-"*65)
    print(f"  순수 개선 편차       |   {metrics_log['auc']-metrics_norm['auc']:+.4f}    |   {metrics_log['acc']-metrics_norm['acc']:+.4f}    |   {metrics_log['rec']-metrics_norm['rec']:+.4f}")
    print("="*65)


# 기술적 의미: 4개 모델 x 2개 실험(A/B) = 8건의 결과를 순서대로 쌓아 둘 빈 리스트를 만든다.
# 업무적 의미: §최종 종합 비교에서 XGBoost 실측치와 합쳐 5개 모델 비교표를 만들 원재료다.
results_log = []


In [13]:
# [담당자: 정찬성 / 모델: RandomForest] 실험 A/B 대조군 학습·평가
# 왜: 배깅(bagging) 계열 대표 모델을 부스팅 계열(XGBoost/LightGBM/GradientBoost) 및 선형모델(LR)과 비교하기 위한 기준점으로 추가한다. §개요 표에서 설명한 대로 max_features/max_samples/min_samples_leaf를 XGBoost의 colsample_bytree/subsample/min_child_weight에 대응시켰다.
from sklearn.ensemble import RandomForestClassifier


# 기술적 의미: 실험마다 완전히 새로운 RandomForestClassifier 객체를 만들어 반환하는 팩토리 함수를 정의한다 — §cell 9의 get_xgb_model()과 동일한 패턴이다.
# 업무적 의미: 실험 A/B가 서로의 학습 상태에 영향을 주지 않도록(독립적인 대조군이 되도록) 매번 새 객체로 시작한다.
def get_rf_model():
    return RandomForestClassifier(
        n_estimators=500, max_depth=5, min_samples_leaf=5,
        max_features=0.8, max_samples=0.8, n_jobs=-1, random_state=156,
    )


print("\n=== RandomForest 비교 학습 및 최종 스코어 측정 ===")

# 기술적 의미: [실험 A] var38 원본 데이터셋(X_train_norm/X_test_norm, §cell 9에서 이미 만든 것)으로 RandomForest를 학습·평가한다.
print("  - 1. [실험 A] var38 원본 데이터셋 기반 모델 학습 중...")
rf_norm, metrics_rf_norm = run_experiment(get_rf_model(), X_train_norm, y_train, X_test_norm, y_test)

# 기술적 의미: [실험 B] var38 로그 변환 데이터셋(X_train_log/X_test_log)으로 RandomForest를 학습·평가한다.
print("  - 2. [실험 B] var38 로그 변환 데이터셋 기반 모델 학습 중...")
rf_log, metrics_rf_log = run_experiment(get_rf_model(), X_train_log, y_train, X_test_log, y_test)

# 기술적 의미: §cell 9와 동일한 서식으로 실험 A/B 성적표를 출력한다.
print_scorecard('RandomForest', metrics_rf_norm, metrics_rf_log)

# 기술적 의미: 실험 A/B 결과를 각각 한 행씩 results_log에 추가한다.
results_log.append({'모델': 'RandomForest', '실험': 'A.원본 var38', **metrics_rf_norm})
results_log.append({'모델': 'RandomForest', '실험': 'B.log1p(var38)', **metrics_rf_log})



=== RandomForest 비교 학습 및 최종 스코어 측정 ===
  - 1. [실험 A] var38 원본 데이터셋 기반 모델 학습 중...
  - 2. [실험 B] var38 로그 변환 데이터셋 기반 모델 학습 중...

        [ RandomForest 최종 분석 논문 검증 결과 종합 성적표 ]
  구분                 |   ROC-AUC   |   정확도    |   재현율   
-----------------------------------------------------------------
  [실험 A] 로그 전    |    0.8414    |    0.9605    |   0.0017
  [실험 B] 로그 후    |    0.8414    |    0.9605    |   0.0017
-----------------------------------------------------------------
  순수 개선 편차       |   +0.0000    |   +0.0000    |   +0.0000


In [14]:
# [담당자: 정찬성 / 모델: LogisticRegression] 실험 A/B 대조군 학습·평가
# 왜: 트리 기반 4개 모델(XGBoost/RandomForest/LightGBM/GradientBoost)과 대비되는 선형모델 기준점(baseline)을 추가한다 — 선형모델은 피처 스케일에 민감하므로, log1p 변환이 트리 모델보다 LR에서 더 뚜렷한 효과를 보일 가능성이 있다(§신용카드 사기검출 노트북에서도 동일한 패턴을 이미 확인한 바 있다).
from sklearn.linear_model import LogisticRegression


# 기술적 의미: 실험마다 새로운 LogisticRegression 객체를 만들어 반환하는 팩토리 함수를 정의한다.
# 업무적 의미: §개요 표에서 설명한 대로, 트리 계열과 달리 맞출 만한 하이퍼파라미터가 없어 max_iter(수렴 보장)만 늘렸다.
def get_lr_model():
    return LogisticRegression(max_iter=1000, random_state=156)


print("\n=== LogisticRegression 비교 학습 및 최종 스코어 측정 ===")

# 기술적 의미: [실험 A] var38 원본 데이터셋으로 LogisticRegression을 학습·평가한다.
print("  - 1. [실험 A] var38 원본 데이터셋 기반 모델 학습 중...")
lr_norm, metrics_lr_norm = run_experiment(get_lr_model(), X_train_norm, y_train, X_test_norm, y_test)

# 기술적 의미: [실험 B] var38 로그 변환 데이터셋으로 LogisticRegression을 학습·평가한다.
print("  - 2. [실험 B] var38 로그 변환 데이터셋 기반 모델 학습 중...")
lr_log, metrics_lr_log = run_experiment(get_lr_model(), X_train_log, y_train, X_test_log, y_test)

# 기술적 의미: 성적표를 출력한다.
print_scorecard('LogisticRegression', metrics_lr_norm, metrics_lr_log)

# 기술적 의미: 실험 A/B 결과를 results_log에 추가한다.
results_log.append({'모델': 'LogisticRegression', '실험': 'A.원본 var38', **metrics_lr_norm})
results_log.append({'모델': 'LogisticRegression', '실험': 'B.log1p(var38)', **metrics_lr_log})



=== LogisticRegression 비교 학습 및 최종 스코어 측정 ===
  - 1. [실험 A] var38 원본 데이터셋 기반 모델 학습 중...
  - 2. [실험 B] var38 로그 변환 데이터셋 기반 모델 학습 중...

        [ LogisticRegression 최종 분석 논문 검증 결과 종합 성적표 ]
  구분                 |   ROC-AUC   |   정확도    |   재현율   
-----------------------------------------------------------------
  [실험 A] 로그 전    |    0.6230    |    0.9603    |   0.0000
  [실험 B] 로그 후    |    0.5473    |    0.9602    |   0.0017
-----------------------------------------------------------------
  순수 개선 편차       |   -0.0757    |   -0.0001    |   +0.0017


In [15]:
# [담당자: 정찬성 / 모델: LightGBM] 실험 A/B 대조군 학습·평가
# 왜: 같은 그레이디언트 부스팅 계열이지만 리프 단위(leaf-wise) 성장 방식을 쓰는 LightGBM을, 레벨 단위 성장인 XGBoost와 동일한 하이퍼파라미터·조기종료 조건으로 비교한다.
from lightgbm import LGBMClassifier
import lightgbm as lgb


# 기술적 의미: 실험마다 새로운 LGBMClassifier 객체를 만들어 반환하는 팩토리 함수를 정의한다. min_child_samples는 XGBoost의 min_child_weight(리프가 되기 위한 최소 조건)에 대응하는 LightGBM 파라미터다. verbose=-1은 학습 중 반복마다 출력되는 내부 로그를 끈다.
def get_lgbm_model():
    return LGBMClassifier(
        n_estimators=500, learning_rate=0.05, max_depth=5, min_child_samples=5,
        subsample=0.8, colsample_bytree=0.8, random_state=156, verbose=-1,
    )


print("\n=== LightGBM 비교 학습 및 최종 스코어 측정 ===")

# 기술적 의미: [실험 A] var38 원본 데이터셋으로 LightGBM을 학습한다. eval_X/eval_y로 검증셋(X_test_norm/y_test)을 지정하고, eval_metric='auc' 기준으로 100라운드 동안 개선이 없으면 멈추는 조기종료 콜백을 건다 — XGBoost의 early_stopping_rounds=100과 동일한 의미다.
print("  - 1. [실험 A] var38 원본 데이터셋 기반 모델 학습 중...")
rf_kwargs_norm = dict(eval_X=X_test_norm, eval_y=y_test, eval_metric='auc',
                       callbacks=[lgb.early_stopping(stopping_rounds=100, verbose=False)])
lgbm_norm, metrics_lgbm_norm = run_experiment(get_lgbm_model(), X_train_norm, y_train, X_test_norm, y_test, **rf_kwargs_norm)

# 기술적 의미: [실험 B] var38 로그 변환 데이터셋으로 동일하게 학습한다. 검증셋도 로그 변환된 X_test_log/y_test로 맞춘다.
print("  - 2. [실험 B] var38 로그 변환 데이터셋 기반 모델 학습 중...")
rf_kwargs_log = dict(eval_X=X_test_log, eval_y=y_test, eval_metric='auc',
                      callbacks=[lgb.early_stopping(stopping_rounds=100, verbose=False)])
lgbm_log, metrics_lgbm_log = run_experiment(get_lgbm_model(), X_train_log, y_train, X_test_log, y_test, **rf_kwargs_log)

# 기술적 의미: 성적표를 출력한다.
print_scorecard('LightGBM', metrics_lgbm_norm, metrics_lgbm_log)

# 기술적 의미: 실험 A/B 결과를 results_log에 추가한다.
results_log.append({'모델': 'LightGBM', '실험': 'A.원본 var38', **metrics_lgbm_norm})
results_log.append({'모델': 'LightGBM', '실험': 'B.log1p(var38)', **metrics_lgbm_log})



=== LightGBM 비교 학습 및 최종 스코어 측정 ===
  - 1. [실험 A] var38 원본 데이터셋 기반 모델 학습 중...
  - 2. [실험 B] var38 로그 변환 데이터셋 기반 모델 학습 중...

        [ LightGBM 최종 분석 논문 검증 결과 종합 성적표 ]
  구분                 |   ROC-AUC   |   정확도    |   재현율   
-----------------------------------------------------------------
  [실험 A] 로그 전    |    0.8490    |    0.9604    |   0.0033
  [실험 B] 로그 후    |    0.8490    |    0.9604    |   0.0033
-----------------------------------------------------------------
  순수 개선 편차       |   +0.0000    |   +0.0000    |   +0.0000


In [16]:
# [담당자: 정찬성 / 모델: GradientBoost] 실험 A/B 대조군 학습·평가
# 왜: XGBoost의 "원조 알고리즘"인 sklearn GradientBoostingClassifier를 최대한 동일한 트리 개수·깊이·학습률·subsample로 평가해, "구현체 최적화(정규화항·히스토그램 분할) 차이"만 순수 비교한다.
from sklearn.ensemble import GradientBoostingClassifier


# 기술적 의미: 실험마다 새로운 GradientBoostingClassifier 객체를 만들어 반환하는 팩토리 함수를 정의한다. n_iter_no_change=100·validation_fraction=0.1은 §개요 표에서 설명한 대로 early_stopping_rounds=100에 대응하는 sklearn 방식의 조기종료 설정이다(학습 데이터의 10%를 내부 검증용으로 떼어 판단).
def get_gb_model():
    return GradientBoostingClassifier(
        n_estimators=500, learning_rate=0.05, max_depth=5, subsample=0.8,
        random_state=156, n_iter_no_change=100, validation_fraction=0.1,
    )


print("\n=== GradientBoost 비교 학습 및 최종 스코어 측정 ===")

# 기술적 의미: [실험 A] var38 원본 데이터셋으로 GradientBoost를 학습·평가한다. GradientBoostingClassifier는 fit()에 eval_set을 받지 않으므로 fit_kwargs 없이 그대로 호출한다.
print("  - 1. [실험 A] var38 원본 데이터셋 기반 모델 학습 중...")
gb_norm, metrics_gb_norm = run_experiment(get_gb_model(), X_train_norm, y_train, X_test_norm, y_test)

# 기술적 의미: [실험 B] var38 로그 변환 데이터셋으로 GradientBoost를 학습·평가한다.
print("  - 2. [실험 B] var38 로그 변환 데이터셋 기반 모델 학습 중...")
gb_log, metrics_gb_log = run_experiment(get_gb_model(), X_train_log, y_train, X_test_log, y_test)

# 기술적 의미: 성적표를 출력한다.
print_scorecard('GradientBoost', metrics_gb_norm, metrics_gb_log)

# 기술적 의미: 실험 A/B 결과를 results_log에 추가한다.
results_log.append({'모델': 'GradientBoost', '실험': 'A.원본 var38', **metrics_gb_norm})
results_log.append({'모델': 'GradientBoost', '실험': 'B.log1p(var38)', **metrics_gb_log})



=== GradientBoost 비교 학습 및 최종 스코어 측정 ===
  - 1. [실험 A] var38 원본 데이터셋 기반 모델 학습 중...
  - 2. [실험 B] var38 로그 변환 데이터셋 기반 모델 학습 중...

        [ GradientBoost 최종 분석 논문 검증 결과 종합 성적표 ]
  구분                 |   ROC-AUC   |   정확도    |   재현율   
-----------------------------------------------------------------
  [실험 A] 로그 전    |    0.8462    |    0.9594    |   0.0150
  [실험 B] 로그 후    |    0.8460    |    0.9592    |   0.0150
-----------------------------------------------------------------
  순수 개선 편차       |   -0.0002    |   -0.0003    |   +0.0000


In [17]:
# [담당자: 정찬성] 실험 B(로그 변환, 4개 모델 공통 최종 학습 결과) 기준 피처 중요도/계수 top10 비교
# 왜: 트리 계열 3종(RandomForest/LightGBM/GradientBoost)의 feature_importances_와 LogisticRegression의 coef_를 나란히 비교해, "어떤 피처가 고객 불만족(TARGET=1)을 가르는 핵심 신호인가"에 대해 계열이 다른 모델들이 같은 결론에 도달하는지 교차검증한다.

# 기술적 의미: RandomForest의 feature_importances_ 상위 10개를 Series로 만들어 출력한다.
print('[RandomForest] 피처 중요도 top10:')
print(pd.Series(rf_log.feature_importances_, index=X_train_log.columns).sort_values(ascending=False).head(10))
print()

# 기술적 의미: LightGBM의 feature_importances_(기본값: 분할 횟수 기준) 상위 10개를 출력한다.
print('[LightGBM] 피처 중요도 top10:')
print(pd.Series(lgbm_log.feature_importances_, index=X_train_log.columns).sort_values(ascending=False).head(10))
print()

# 기술적 의미: GradientBoost의 feature_importances_ 상위 10개를 출력한다.
print('[GradientBoost] 피처 중요도 top10:')
print(pd.Series(gb_log.feature_importances_, index=X_train_log.columns).sort_values(ascending=False).head(10))
print()

# 기술적 의미: LogisticRegression은 계수(coef_) 절댓값 상위 10개를 부호(+/-)와 함께 출력한다.
print('[LogisticRegression] |계수| top10(부호 포함):')
lr_coef = pd.Series(lr_log.coef_[0], index=X_train_log.columns)
print(lr_coef.reindex(lr_coef.abs().sort_values(ascending=False).head(10).index))


[RandomForest] 피처 중요도 top10:
var15                      0.388424
saldo_var30                0.249471
var38                      0.115509
ind_var30                  0.018653
num_var30                  0.018139
imp_op_var41_efect_ult1    0.012884
saldo_medio_var5_hace3     0.009115
imp_op_var39_efect_ult1    0.008691
num_var22_ult3             0.008254
saldo_var42                0.007890
dtype: float64

[LightGBM] 피처 중요도 top10:
var38                     239
var15                     224
saldo_var30               128
saldo_medio_var5_ult3     115
saldo_medio_var5_hace2     99
saldo_medio_var5_hace3     98
num_var22_ult3             74
num_var45_hace3            73
num_var22_hace3            65
saldo_var5                 62
dtype: int32

[GradientBoost] 피처 중요도 top10:
var15                      0.222364
saldo_var30                0.180179
var38                      0.137830
saldo_medio_var5_hace3     0.026263
saldo_medio_var5_hace2     0.016938
saldo_medio_var5_ult3      0.014922
saldo_var5

---

## [담당자: 정찬성] 최종 종합 비교 — 5개 모델 x 실험 A/B

```mermaid
flowchart LR
    A["results_log(8건)<br/>RandomForest/LogisticRegression/LightGBM/GradientBoost x 실험 A/B"] --> B["XGBoost 실측치 2건<br/>(§cell 9 결과 주석에서 가져옴)"]
    B --> C["5개 모델 x 2실험 = 10행<br/>종합 비교 DataFrame"]
    C --> D["실험별 pivot<br/>(AUC 기준 모델 랭킹)"]
```

In [18]:
# 기술적 의미: results_log(신규 4개 모델 x 2개 실험 = 8건)를 pandas DataFrame으로 변환한다.
new_models_df = pd.DataFrame(results_log)

# 기술적 의미: XGBoost는 이미 위(§cell 9)에서 실제로 실행돼 "내 결과" 주석으로 실측치가 기록돼 있다 — 그 값을 그대로 옮겨 적어(재실행하지 않고) 나머지 4개 모델과 같은 형식의 행으로 만든다.
# 업무적 의미: 이미 신뢰할 수 있는 값을 다시 계산하느라 시간을 쓰지 않고, 기존 실행 로그(§cell 9)를 "단일 진실 공급원"으로 재사용한다.
xgb_summary = pd.DataFrame([
    {'모델': 'XGBoost', '실험': 'A.원본 var38', 'auc': 0.8520, 'acc': 0.9606, 'rec': 0.0066},
    {'모델': 'XGBoost', '실험': 'B.log1p(var38)', 'auc': 0.8520, 'acc': 0.9606, 'rec': 0.0066},
])

# 기술적 의미: 신규 4개 모델 결과(new_models_df)와 XGBoost 실측치(xgb_summary)를 세로로 이어붙여 5개 모델 x 2실험 = 10행짜리 종합 비교표를 만든다.
full_comparison = pd.concat([xgb_summary, new_models_df], ignore_index=True)

# 기술적 의미: 종합 비교표 전체를 출력한다.
print('=== 5개 모델 x 실험 A/B 종합 비교 ===')
print(full_comparison.to_string(index=False))
print()

# 기술적 의미: 실험(A/B)을 행, 모델을 열로 하고 AUC 값을 채운 pivot 표를 만든다.
# 업무적 의미: "이 실험 조건에서는 어느 모델이 가장 판별력이 좋았는가"를 한눈에 가로 비교할 수 있게 한다.
auc_pivot = full_comparison.pivot(index='실험', columns='모델', values='auc')

# 기술적 의미: AUC 기준 pivot 표를 출력한다.
print('=== 실험별 AUC 비교(모델을 열로) ===')
print(auc_pivot.to_string())


=== 5개 모델 x 실험 A/B 종합 비교 ===
                모델             실험      auc      acc      rec
           XGBoost     A.원본 var38 0.852000 0.960600 0.006600
           XGBoost B.log1p(var38) 0.852000 0.960600 0.006600
      RandomForest     A.원본 var38 0.841409 0.960471 0.001661
      RandomForest B.log1p(var38) 0.841409 0.960471 0.001661
LogisticRegression     A.원본 var38 0.623006 0.960339 0.000000
LogisticRegression B.log1p(var38) 0.547337 0.960208 0.001661
          LightGBM     A.원본 var38 0.848967 0.960405 0.003322
          LightGBM B.log1p(var38) 0.848967 0.960405 0.003322
     GradientBoost     A.원본 var38 0.846234 0.959419 0.014950
     GradientBoost B.log1p(var38) 0.845987 0.959155 0.014950

=== 실험별 AUC 비교(모델을 열로) ===
모델              GradientBoost  LightGBM  LogisticRegression  RandomForest  XGBoost
실험                                                                                
A.원본 var38           0.846234  0.848967            0.623006      0.841409    0.852
B.log1p(var38)       0.

### 최종 해석

> ⏸ 이 셀은 실행 대기 상태다 — 위 종합 비교표를 실제로 생성한 뒤, 아래 항목을 채운다.

- **부스팅 3종(XGBoost/LightGBM/GradientBoost) 간 비교**: 동일한 트리 개수·깊이·학습률·subsample로 맞췄을 때, AUC·재현율이 어느 구현체에서 가장 좋은지 확인할 것. XGBoost는 실험 A/B 모두 재현율이 0.0066로 극히 낮았다 — 이는 클래스 불균형(TARGET=1 비율 약 3.96%, §cell 2)에서 기본 임계값(0.5)이 소수 클래스를 거의 잡아내지 못한다는 신호이며, 나머지 3개 부스팅 모델도 같은 문제를 보이는지, 혹은 조기종료·리프 성장 방식 차이로 이 문제가 완화되는지가 핵심 확인 포인트다.
- **배깅(RandomForest) vs 부스팅 3종**: RandomForest가 재현율 측면에서 부스팅 계열보다 나은지(배깅은 부스팅과 달리 오분류에 가중치를 주는 메커니즘이 없어 불균형에 더 취약할 수도, 트리 다양성 덕에 더 강건할 수도 있다) 실측으로 확인할 것.
- **선형(LogisticRegression) vs 트리 4종**: LR의 AUC가 XGBoost(0.8520)에 근접한다면, 이 데이터의 신용 위험 신호 상당 부분이 선형적으로 분리 가능하다는 뜻이고, 재현율이 유의미하게 다르다면 결정 경계의 형태(선형 vs 비선형) 차이가 실제로 드러난 것이다.
- **var38 로그 변환의 효과**: XGBoost는 실험 A/B가 완전히 동일했다("트리 기반 모델은 단조 변환에 불변") — RandomForest/LightGBM/GradientBoost도 같은 불변성을 보이는지, 반대로 스케일에 민감한 LogisticRegression에서는 실제로 유의미한 차이가 나는지 5개 모델 전체로 검증할 것.
- **극단적으로 낮은 재현율에 대한 권고**: 5개 모델 모두 재현율이 낮게 나온다면, 이 노트북의 범위를 벗어나는 후속 과제로 `class_weight='balanced'`, 임계값(threshold) 조정, 또는 신용카드 사기검출 노트북(§99_2)에서 이미 검증한 SMOTE 오버샘플링 적용을 검토 대상으로 제안할 수 있다.